Code-only archive: embedded input/reference bundles were omitted. Run the adjacent build_notebook.py or build_notebooks.py with your local inputs to recreate the executable Kaggle notebook.


# PROTEUS: author iTransformer without lags vs single Lag4

Private GPU experiment;no competition submission. Article-aligned8LOSO,later-session and old public test.

**iTransformer:** official pinned encoder c2426e68;41tokens of480samples;one block,d64,4heads,FFN128,GELU,dropout0.2;flatten+linear3-class head. AdamW0.001,decay0.001,batch64,max80epochs,patience10. Seeds0,1,2 reported separately.

**Baseline:** exact single Lag4 OAS+causal reference+LR;41electrodes expanded to82rows,prior5,C0.01.

Inner validation people never enter fitting or normalization. Best epoch selected on inner validation weighted CE;fresh refit on all outer-training people. Curves below update during inner selection;outer evaluation never appears as validation.

In [ ]:
import os,sys,pathlib,json,hashlib,zipfile,io,base64,subprocess
os.environ.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",PYTHONUNBUFFERED="1",MPLCONFIGDIR="/tmp/it-mpl")
import torch
assert torch.cuda.is_available(),"Enable requested Kaggle GPU"
print("GPU",torch.cuda.get_device_name(0),flush=True)
project=pathlib.Path("/kaggle/working/riemann-project");project.mkdir(exist_ok=True)
sources=list(pathlib.Path("/kaggle/input").rglob("source-integrity.json"));assert len(sources)==1
raw=sources[0].read_bytes()
assert hashlib.sha256(raw).hexdigest()=="3e8cf0b952a1a18ab496a89d6a31275522e8efa003ea6b8ed4b9e6467d2a3c81"
for rel,digest in json.loads(raw).items():
    p=pathlib.PurePosixPath(rel);assert not p.is_absolute() and ".." not in p.parts
    data=(sources[0].parent/rel).read_bytes();assert hashlib.sha256(data).hexdigest()==digest
    dest=project/rel;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
payload=base64.b64decode(__import__("pathlib").Path("local-bundle.base64").read_text())
assert hashlib.sha256(payload).hexdigest()=="417c358c4a07c53e0c35d5271251e8d66168296078594a960147a8363b7493fc"
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    manifest=json.loads(z.read("it-manifest.json"))
    for rel,digest in manifest.items():
        p=pathlib.PurePosixPath(rel);assert not p.is_absolute() and ".." not in p.parts
        data=z.read(rel);assert hashlib.sha256(data).hexdigest()==digest
        dest=project/rel;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
repro=project/"reproduction";experiment=repro/"itransformer_ablation"
inputs=list(pathlib.Path("/kaggle/input").rglob("input-integrity.json"));assert len(inputs)==1
input_manifest=json.loads(inputs[0].read_text());input_dir=inputs[0].parent
for name,digest in input_manifest["files_sha256"].items():
    h=hashlib.sha256()
    with (input_dir/name).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    assert h.hexdigest()==digest,name
(repro/"data").mkdir(exist_ok=True)
for split in ["train","val","test"]:
    (repro/"data"/f"proteus_{split}.npz").symlink_to(input_dir/f"proteus_{split}.npz")
(repro/"temporal_spd_research/cache").symlink_to(input_dir,target_is_directory=True)
envdir=pathlib.Path("/tmp/it-env")
subprocess.run([sys.executable,"-m","venv","--without-pip","--system-site-packages",str(envdir)],check=True)
python=str(envdir/"bin/python")
pins=["numpy==2.5.4","scipy==1.18.1","scikit-learn==1.9.1","pyriemann==0.12","pandas==3.1.0","threadpoolctl==3.7.0","einops==0.8.2"]
subprocess.run([sys.executable,"-m","pip","--python",python,"install","--quiet",*pins],check=True)
out=pathlib.Path("/kaggle/working/it-results");out.mkdir(exist_ok=True)
env=dict(os.environ,IT_OUTPUT=str(out))
(out/"bundle-provenance.json").write_text(json.dumps(dict(it_bundle_sha256="417c358c4a07c53e0c35d5271251e8d66168296078594a960147a8363b7493fc",original_source_manifest_sha256=hashlib.sha256(raw).hexdigest(),input_manifest=input_manifest,source_manifest=manifest),indent=2))
print("Verified pinned author iTransformer and inputs.41channels,480samples,NO lag copies;single Lag4 baseline.",flush=True)


## Verify author equivalence,split isolation,checkpoint selection and integrity

In [ ]:
subprocess.run([python,"-m","pytest","-q","--tb=short"],cwd=experiment,env=env,check=True)

## Fit models and show train/validation curves

Each figure is an inner person-held-out fit. Blue=train,orange=validation,both in eval mode. Dotted lines show the inner Lag4 baseline;vertical marker shows selected epoch. CSV saved each epoch;PNG updated every5epochs and at completion. Fresh outer refit has its own training-history CSV.

In [ ]:
from IPython.display import display, Image
figures={}
process=subprocess.Popen([python,"-u",str(experiment/"run_itransformer.py")],cwd=experiment,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:
    print(line,end="",flush=True)
    if line.startswith("CURVE "):
        figure=pathlib.Path(line.strip()[6:])
        if figure.exists():
            picture=Image(filename=str(figure))
            if str(figure) in figures:figures[str(figure)].update(picture)
            else:figures[str(figure)]=display(picture,display_id=True)
code=process.wait()
if code:raise RuntimeError(f"Experiment exited with status {code};partial curves and checkpoints are saved")


## Independent metric verification and final comparison

In [ ]:
subprocess.run([python,"-u",str(experiment/"analyze_itransformer.py")],cwd=experiment,env=env,check=True)
print((out/"REPORT.md").read_text())